# 07 · ClinVar — the crowd-sourced clinical truth set

A predictor's number is meaningless until you compare it against a **truth set** — variants whose clinical status someone already established. **ClinVar** is the public archive of clinical assertions from many labs. This data is **REAL** (genuine CFTR ClinVar assertions); every row reads `source == 'REAL'`.

> ✅ **REAL data.** These are genuine ClinVar assertions for *CFTR*, loaded from the cached ClinVar extract. Notebook 08 adds the second truth set (CFTR2); the archived integration notebook cross-checks the two.

In [1]:
import sys, pathlib
# `toolkit` is THIS repo's toolkit.py (one directory up) — NOT a pip
# package and nothing to do with gnomAD. The line below puts the repo
# root on sys.path so `import toolkit` resolves to ../toolkit.py.
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import toolkit as tk
import pandas as pd, numpy as np
# %matplotlib inline is a Jupyter magic: it draws matplotlib plots inline below the cell
%matplotlib inline

## 1 · ClinVar — what is it?

[ClinVar](https://www.ncbi.nlm.nih.gov/clinvar/) is a free, public archive run by the NIH. Clinical genetics labs around the world **submit their interpretation** of a variant: is it disease-causing or not? ClinVar aggregates those submissions and reports a clinical significance, usually one of:

- **Pathogenic** / **Likely pathogenic**
- **Uncertain significance** (a *VUS* — Variant of Uncertain Significance)
- **Likely benign** / **Benign**
- **Conflicting classifications** (labs disagreed)

Think of it as a giant, crowd-sourced ledger of *clinical opinion*. Its great strength is **breadth** — thousands of CFTR variants. Its weakness is that it is **heterogeneous**: different submitters, different rigor, different years. Let's load it.

> **How this data is fetched & pinned.** `load_clinvar()` reads a cached extract that `fetch_scores.py::fetch_clinvar_cftr` built from NCBI's **`variant_summary.txt.gz`** (bulk FTP, `ftp.ncbi.nlm.nih.gov/pub/clinvar/`, updated **~weekly**), filtered to CFTR. Because ClinVar changes constantly, **pin the exact release date** in `data_manifest.json` — the pinned release decides which variants count as VUS (and therefore the A1 headline numbers). Ref: Landrum et al., *Nucleic Acids Res*.

In [2]:
clinvar = tk.load_clinvar()
print('rows:', len(clinvar))
print('source values:', clinvar['source'].unique(), ' <- REAL data')
clinvar.head()

rows: 2801
source values: ['REAL']  <- REAL data


,protein_variant,clinvar_sig,review_status,clinvar_call,source
0,Q493*,Pathogenic,reviewed by expert panel,pathogenic,REAL
1,D110H,Pathogenic; drug response,reviewed by expert panel,pathogenic,REAL
2,R117H,Pathogenic,practice guideline,pathogenic,REAL
3,R347P,Pathogenic,practice guideline,pathogenic,REAL
4,A455E,Pathogenic,practice guideline,pathogenic,REAL


Every row is `source == 'REAL'`: these are genuine ClinVar assertions for *CFTR*, not something the toolkit made up. Each row has:
- `protein_variant` — the amino-acid change (e.g. `G551D`)
- `clinvar_sig` — the raw, free-text clinical significance
- `review_status` — **how much confidence** to place in it (next section)
- `clinvar_call` — a tidy 3-class collapse of `clinvar_sig`

## 2 · Review status — the gold stars (⭐) that matter most

**This is the single most important slide in the notebook.** Not every ClinVar assertion deserves equal trust. ClinVar attaches a **review status**, which maps to a **0–4 gold-star** rating:

| Stars | Review status | Roughly means |
|:--:|---|---|
| 0 ⭐ | *no assertion criteria provided* | someone's opinion, no stated method |
| 1 ⭐ | *criteria provided, single submitter* | one lab, with a method |
| 1 ⭐ | *criteria provided, conflicting classifications* | labs disagree |
| 2 ⭐ | *criteria provided, multiple submitters, no conflicts* | several labs agree |
| 3 ⭐ | *reviewed by expert panel* | e.g. the CFTR2 / ClinGen expert panel |
| 4 ⭐ | *practice guideline* | encoded in clinical practice guidelines |

Let's count how our CFTR variants are distributed across these levels.

In [3]:
clinvar['review_status'].value_counts()

review_status
criteria provided, single submitter                     1373
criteria provided, multiple submitters, no conflicts     788
criteria provided, conflicting classifications           229
reviewed by expert panel                                 196
no assertion criteria provided                           145
no classification provided                                56
practice guideline                                        12
-                                                          2
Name: count, dtype: int64

**Read this carefully.** The largest bucket is *single submitter* (1 star) — one lab's call. Only a couple hundred are *expert panel* (3 stars), and a dozen are *practice guideline* (4 stars).

> **KEY LESSON:** a **1-star VUS** and a **3-star expert-panel Pathogenic** are *not* the same quality of evidence, even though both are 'in ClinVar'. When you build a benchmark, you often **filter to ≥2 stars** so a predictor is judged against assertions people actually trust — not against one lab's unreviewed guess. Throwing every star level into the same pile is one of the most common ways people accidentally lie to themselves with ClinVar.

## 3 · The tidy 3-class call — and why VUS are the whole point

The raw `clinvar_sig` text is messy (`'Pathogenic/Likely pathogenic'`, `'Conflicting classifications of pathogenicity'`, …). The toolkit collapses it into three clean buckets in `clinvar_call`. Let's count them.

In [4]:
counts = clinvar['clinvar_call'].value_counts()
print(counts)
n_vus = int(counts.get('uncertain', 0))
print(f'\nUncertain / VUS: {n_vus}  ({100*n_vus/len(clinvar):.0f}% of all rows)')

clinvar_call
uncertain     2246
pathogenic     540
benign          15
Name: count, dtype: int64

Uncertain / VUS: 2246  (80% of all rows)


Notice how **most CFTR variants in ClinVar are `uncertain` (VUS)**. That is not a bug — it is the *reason predictors exist*. If every variant already had a confident Pathogenic/Benign label, we would not need EVE or AlphaMissense. The huge pile of VUS is precisely the set of variants a lab gets back from sequencing and *cannot act on*. A good predictor's job is to help **re-classify** those VUS — so they are the interesting, unsolved cases, not the easy ones.

## 4 · How `cv_class` collapses the free text (and one deliberate choice)

`tk.cv_class()` is the little function that turns messy significance text into `pathogenic` / `benign` / `uncertain`. Let's watch it work on a few example strings.

In [5]:
examples = [
    'Pathogenic',
    'Benign/Likely benign',
    'Uncertain significance',
    'Conflicting classifications of pathogenicity',
]
pd.DataFrame({
    'input_significance': examples,
    'cv_class_output':    [tk.cv_class(s) for s in examples],
})

,input_significance,cv_class_output
0,Pathogenic,pathogenic
1,Benign/Likely benign,benign
2,Uncertain significance,uncertain
3,Conflicting classifications of pathogenicity,uncertain


Three of these are obvious. The fourth is the interesting one:

> **`Conflicting classifications` → `uncertain`.**

When labs *disagree* about a variant, the toolkit deliberately does **not** guess a winner — it treats the variant as `uncertain`. Why is that the *conservative* and correct choice for benchmarking?

- If we forced a conflicting variant into `pathogenic` or `benign`, we would be inventing a ground-truth label that **doesn't exist** — the experts themselves couldn't agree.
- Scoring a predictor against a made-up label would make the benchmark look cleaner than reality and could unfairly reward *or* punish the tool.

Collapsing conflicts to `uncertain` keeps them out of the Pathogenic-vs-Benign scoring, which is exactly where you want honest, high-confidence labels.

## The shared A1 worked-example panel — **ClinVar** in context

The same ~14 famous CFTR **missense** variants are scored by *every* A1 tool
(notebooks 01–08), so you can follow one fixed panel across the whole series. Your
column here is **`clinvar_call`**; the CFTR2 / ClinVar truth columns are shown for context.

`tk.a1_panel()` is the single source of truth (defined in `toolkit.py`); it uses the
REAL extracts where present (missing extracts → NaN).

In [6]:
tk.a1_panel()

,protein_variant,gnomad_af,AM,EVE,ESM1b,REVEL,PAI,cftr2_class,clinvar_call
0,G551D,0.000404,0.9897,0.939098,-12.123,0.990,0.757039,CF-causing,pathogenic
1,F508del,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,R117H,0.002199,0.2985,0.819590,-5.659,0.807,0.538583,Varying clinical consequence,pathogenic
3,R334W,0.000092,0.6563,0.950636,-8.179,0.816,0.715262,CF-causing,pathogenic
4,G85E,0.000065,0.9881,0.922779,-11.889,0.940,0.775947,CF-causing,pathogenic
5,D1152H,0.000407,0.8597,0.842551,-9.784,0.657,0.687562,Varying clinical consequence,uncertain
6,R668C,0.008450,0.1040,0.921812,-9.968,0.706,0.713809,Non CF-causing,uncertain
7,Y161C,0.000002,0.8006,0.934065,-8.857,0.969,0.844116,No interpretation available,uncertain
8,G970D,0.000007,0.7638,0.930015,-12.000,0.985,0.621215,CF-causing,pathogenic
9,S912L,0.000663,0.1245,0.085494,-3.418,0.543,0.301304,Varying clinical consequence,uncertain


## Key takeaways

1. **ClinVar** aggregates clinical assertions from many submitters; **review status** (the gold stars) tells you how much to trust each one.
2. Most *CFTR* ClinVar variants are **uncertain (VUS)** — that backlog is the whole point of the triage.
3. ClinVar is **REAL** here. But it is *crowd-sourced clinical opinion* — notebook 08 adds **CFTR2**, a more *functional* and orthogonal truth set.

**Next:** notebook 08 — **CFTR2**.